####  Extending P0 with Bitwise Set Operations (WASM)

In [1]:
def runwasm(wasmfile):
  from IPython.display import display, Javascript

  with open(wasmfile, 'rb') as file:
    wasm_bytes = file.read()  # read the WASM file and convert its content to a comma-separated string of byte values
    wasm_byte_array_str = ','.join(str(byte) for byte in wasm_bytes)

  display(
    Javascript(f""" // Javascript code to compile and run the WASM module
    const wasmBinary = new Uint8Array([{wasm_byte_array_str}]); // convert back to binary representation
 
    const params = {{
        'P0lib': {{
            'write': i => element.append(i + ' '),
            'writeln': () => element.append(document.createElement('br')),
            'read': () => window.prompt()
        }}
    }};
 
    WebAssembly.compile(wasmBinary.buffer) // compile shareable code
        .then(module => WebAssembly.instantiate(module, params)) // create an instance with memory
        // .then(instance => instance.exports.program()); // run the main program; not needed if a start function is specified
    """)
  )

In [2]:
def runpywasm(wasmfile):
  from pywasm.core import Machine, Runtime, FuncType, ValType

  # P0lib implementation in Python
  def write(_: Machine, args: list[int]) -> list[int]:
    print(args[0])
    return []

  def writeln(_: Machine, args: list[int]) -> list[int]:
    print()
    return []

  def read(_: Machine, args: list[int]) -> list[int]:
    return [int(input())]

  # Create runtime
  runtime = Runtime()
  runtime.imports = {
    'P0lib': {
      'write': runtime.allocate_func_host(FuncType([ValType.i32()], []), write),
      'writeln': runtime.allocate_func_host(FuncType([], []), writeln),
      'read': runtime.allocate_func_host(FuncType([], [ValType.i32()]), read),
    }
  }
  # Create and run instance
  instance = runtime.instance_from_file(wasmfile)

In [3]:
def runwasmtime(wasmfile):
  from wasmtime import Engine, Store, Module, Linker, Func, FuncType, ValType

  # P0lib implementation in Python
  def write(i: int):
    print(i)

  def writeln():
    print()

  def read() -> int:
    return int(input())

  # Create engine, store, and module
  engine = Engine()
  store = Store(engine)
  module = Module(store.engine, open(wasmfile, 'rb').read())
  # Use Linker to create the P0lib library
  write_func = Func(store, FuncType([ValType.i32()], []), write)
  writeln_func = Func(store, FuncType([], []), writeln)
  read_func = Func(store, FuncType([], [ValType.i32()]), read)
  linker = Linker(engine)
  linker.define(store, 'P0lib', 'write', write_func)
  linker.define(store, 'P0lib', 'writeln', writeln_func)
  linker.define(store, 'P0lib', 'read', read_func)
  # Create and run instance
  instance = linker.instantiate(store, module)

In [4]:
import import_ipynb
from P0 import compileString

Extend P0 with two binary operators, `∉` (Unicode U+2209, not an element of) and `∖` (Unicode U+2216, set minus), defined by:
* `i ∉ s ≡ ¬(i ∈ s)`, where `i` is an integer and `s` is a set 
* `s ∖ t = s ∩ ∁t`, where `s`, `t` are sets

The following P0 program illustrates the use of these operators; you can use it to test your implementation:

In [5]:
compileString(
  """
const N = 32
const R = 5 // ⌊√N⌋
type S = set [0 .. N - 1]
procedure eratosthenes() → (p: S)
    var i, j: integer
        p := ∁ {} // set of all integers 
        i := 2;
        while i ≤ R do
            if i ∈ p then
                j := i × i
                while j < N do
                    p, j := p ∖ {j}, j + i
            i := i + 1
program primes
    var p: S
    var j: integer
        p ← eratosthenes()
        j := 2 // print all primes up to N
        while j < N do
            if j ∈ p then write(j)
            j := j + 1
        writeln(); j := 2 // print all non-primes up to N
        while j < N do
            if j ∉ p then write(j)
            j := j + 1
""",
  'primes.wat',
  target='wat',
)

In [6]:
!wat2wasm primes.wat

In [7]:
runpywasm('primes.wasm')

2
3
5
7
11
13
17
19
23
29
31

4
6
8
9
10
12
14
15
16
18
20
21
22
24
25
26
27
28
30


The output is supposed to be:
```
2 3 5 7 11 13 17 19 23 29 31 
4 6 8 9 10 12 14 15 16 18 20 21 22 24 25 26 27 28 30
```

1. Extending the Scanner

- In `SC.ipynb`, introduce new integer constants `NOTELEMENT` and `DIFFERENCE` for `∉` and `∖`.
- Extend the production of `symbol` in the text cell above `getSym()` to include  `∉` and `∖`.
- Extend `getSym()` to return `NOTELEMENT` and `DIFFERENCE` when recognizing  `∉` and `∖`.

The cell below serves only for grading and testing.

In [8]:
import SC


def scanString(src):
  SC.init(src)
  syms = [(SC.sym, SC.val)]
  while SC.sym != SC.EOF:
    SC.getSym()
    syms.append((SC.sym, SC.val))
  return syms


assert scanString('5 ∉ {3}') == [
  (SC.NUMBER, 5),
  (SC.NOTELEMENT, 5),
  (SC.LBRACE, 5),
  (SC.NUMBER, 3),
  (SC.RBRACE, 3),
  (SC.EOF, 3),
]
assert scanString('s ∖ t') == [(SC.IDENT, 's'), (SC.DIFFERENCE, 's'), (SC.IDENT, 't'), (SC.EOF, 't')]

2. Extending the Parser

1. In `P0.ipynb`, extend the imports with `NOTELEMENT` and `DIFFERENCE`.
2. Extend the grammar at two places, in the section "The P0 Grammar" and before the parsing procedures for each nonterminal: `∖` has to bind as tight as `∩` and `∉` has to bind as tight as `∈`. That is, `a ∉ b ∖ c ∪ d` is parsed like `a ∉ ((b ∖ c) ∪ d)`.
3. If the first sets need to be modified, modify them.
4. Extend the parsing procedures accordingly. Add type-checking: when parsing `a ∉ b`, if `a` is not an integer, the error `bad type` should be generated; if `b` is not a set, the error `set expected` should be generated. When parsing `a ∖ b`, the error `bad type` should be generated if not both `a` and `b` are sets.
5. In `CGast.ipynb`, extend the imports with `NOTELEMENT` and `DIFFERENCE`.
6. Extend method `__str(self)__` of class `BinaryOp` such that `∉` is returned for `NOTELEMENT` and `∖` is returned for `DIFFERENCE`.

The cell below serves only for grading and testing.

In [9]:
assert (
  compileString(
    """
program bitsets
  var a, b: set [1 .. 5]
  var c: boolean
    c := (2 ∈ a) or (2 ∉ a)
    a := ∁ a ∩ b ∖ a ∪ b
""",
    target='ast',
  )
  == """\
seq
  :=
    Var(name = c, lev = 1, tp = <class 'ST.Bool'>)
    or
      ∈
        Const(name = , tp = <class 'ST.Int'>, val = 2)
        Var(name = a, lev = 1, tp = Set(lower = 1, length = 5))
      ∉
        Const(name = , tp = <class 'ST.Int'>, val = 2)
        Var(name = a, lev = 1, tp = Set(lower = 1, length = 5))
  :=
    Var(name = a, lev = 1, tp = Set(lower = 1, length = 5))
    ∪
      ∖
        ∩
          ∁
            Var(name = a, lev = 1, tp = Set(lower = 1, length = 5))
          Var(name = b, lev = 1, tp = Set(lower = 1, length = 5))
        Var(name = a, lev = 1, tp = Set(lower = 1, length = 5))
      Var(name = b, lev = 1, tp = Set(lower = 1, length = 5))"""
)

The following programs should all produce error messages:

In [ ]:
try:
  compileString(
    """
program bitsets
  var a, b: set [1 .. 5]
  var c: boolean
    c := a ∉ b
""",
    target='ast',
  )
  raise
except Exception as e:
  assert str(e) == 'line 5 pos 14 bad type'

In [ ]:
try:
  compileString(
    """
program bitsets
  var a, b: set [1 .. 5]
  var c: boolean
    c := 2 ∉ c
""",
    target='ast',
  )
  raise
except Exception as e:
  assert str(e) == 'line 5 pos 14 set expected'

In [ ]:
try:
  compileString(
    """
program bitsets
  var a, b: set [1 .. 5]
  var c: boolean
    a := c ∖ b
""",
    target='ast',
  )
  raise
except Exception as e:
  assert str(e) == 'line 5 pos 14 bad type'

In [ ]:
try:
  compileString(
    """
program bitsets
  var a, b: set [1 .. 5]
    a := b ∖ 3
""",
    target='ast',
  )
  raise
except Exception as e:
  assert str(e) == 'line 4 pos 14 bad type'

4. Extending the Code Generator

In `CGwat.ipynb`, extend `genUnaryOp` and `genBinaryOp` to generate code for `∉` and `∖`.

The cell below serves only for grading and testing. *Hint:* when generating code for `a ∖ b` in `genBinaryOp`, load first `b` on the stack, complement it, then load `a` on the stack and bitwise-and the top two elements on the stack.

In [ ]:
import nbimporter

nbimporter.options['only_defs'] = False
import P0

compileString(
  """
program bitsets
  var a, b: set [1 .. 11]
  var i: integer
    i, a := 1, {3, 5, 7}
    if i ∉ a then a := a ∖ {7, 9}
    while i < 12 do
      if i ∈ a then write(i)
      i := i + 1
""",
  'bitsets.wat',
  target='wat',
)

assert (
  open('bitsets.wat').read()
  == """\
(module
(import "P0lib" "write" (func $write (param i32)))
(import "P0lib" "writeln" (func $writeln))
(import "P0lib" "read" (func $read (result i32)))
(global $_memsize (mut i32) i32.const 0)
(func $program
(local $a i32)
(local $b i32)
(local $i i32)
(local $0 i32)
i32.const 1
i32.const 3
local.set $0
i32.const 1
local.get $0
i32.shl
i32.const 5
local.set $0
i32.const 1
local.get $0
i32.shl
i32.or
i32.const 7
local.set $0
i32.const 1
local.get $0
i32.shl
i32.or
local.set $a
local.set $i
local.get $i
local.set $0
i32.const 1
local.get $0
i32.shl
local.get $a
i32.and
i32.eqz
if
i32.const 7
local.set $0
i32.const 1
local.get $0
i32.shl
i32.const 9
local.set $0
i32.const 1
local.get $0
i32.shl
i32.or
i32.const 0xffe
i32.xor
local.get $a
i32.and
local.set $a
end
loop
local.get $i
i32.const 12
i32.lt_s
if
local.get $i
local.set $0
i32.const 1
local.get $0
i32.shl
local.get $a
i32.and
if
local.get $i
call $write
end
local.get $i
i32.const 1
i32.add
local.set $i
br 1
end
end
)
(memory 1)
(start $program)
)"""
)

The first if-statement should translate to:
```
local.get $i
local.set $0
i32.const 1
local.get $0
i32.shl
local.get $a
i32.and
i32.eqz
if
i32.const 7
local.set $0
i32.const 1
local.get $0
i32.shl
i32.const 9
local.set $0
i32.const 1
local.get $0
i32.shl
i32.or
i32.const 0xffe
i32.xor
local.get $a
i32.and
local.set $a
end
```

Running the program should print `3 5`:

In [ ]:
!wat2wasm bitsets.wat

In [ ]:
runpywasm('bitsets.wasm')

4. Evaluating the Implementation

The task is to compare the efficiency of 4 implementations of the Sieve of Eratosthenes with sets:
- P0 using pywasm, the Python interpreter of WebAssembly,
- P0 using the JavaScript host of WebAssembly in your web browser,
- Python using the standard CPython implementation,
- Java using the standard Oracle JVM.

The P0, Python, and Java implementations are already provided. To keep it simple, the [Jupyter time cell magic](https://ipython.readthedocs.io/en/stable/interactive/magics.html) is used rather than printing the start and end time within the programs. That command measures the *wall clock time*. This means that the time for loading and compilation is included in the measured time. We, therefore, aim at execution times above 1 second to make the time for loading and compiling negligible. For this, the Sieve of Eratosthenes is repeatedly run. The times will vary with each run. Longer times typically result from an unrelated CPU load and should be disregarded. The best is to run the code at times of the day when there is as little interference as possible. Vary the number of repetitions. Document your observations and explain!  

YOUR ANSWER HERE

In [ ]:
compileString(
  """
const N = 32
const R = 5 // ⌊√N⌋
type S = set [2 .. 31]
procedure eratosthenes() → (p: S)
    var i, j: integer
        p := ∁ {} // set of all integers 
        i := 2;
        while i ≤ R do
            if i ∈ p then
                j := i × i
                while j < N do
                    p, j := p ∖ {j}, j + i
            i := i + 1
program repeatprimes
    var p: S
    var j: integer
        j := 0
        while j < 1000 do
            p ← eratosthenes(); j := j + 1
""",
  'repeatprimes.wat',
  target='wat',
)

In [ ]:
!wat2wasm repeatprimes.wat

In [ ]:
time runpywasm("repeatprimes.wasm")

In [ ]:
time runwasm("repeatprimes.wasm")

In [ ]:
time runwasmtime("repeatprimes.wasm")

In [ ]:
def eratosthenes():
  N, R = 32, 5
  p = {i for i in range(2, N)}
  for i in range(2, R):
    if i in p:
      for j in range(i * i, N, i):
        p -= {j}
  return p


def repeatprimes():
  for _ in range(1000000):
    eratosthenes()

In [ ]:
time repeatprimes()

In [ ]:
%%writefile repeatprimes.java
import java.util.*;
class RepeatPrimes {
    static final int N = 32;
    static final int R = 5;
    static Set<Integer> eratosthenes() {
        Set<Integer> p = new HashSet<Integer>();
        for (int i = 2; i < N; i++) p.add(i);
        for (int i = 2; i < R; i++)
            if (p.contains(i))
                for (int j = i * i; j < N; j = j + i)
                    p.remove(j);
        return p;
    }
    public static void main(String[] args) {
        for (int i = 0; i < 100000000; i++) eratosthenes();
    }
}

In [ ]:
!javac repeatprimes.java

In [ ]:
time !java RepeatPrimes